# Error Services
This notebook can be used to test both the Simple- and Continuous Error Services.

If all services are not already started, open a shell and run:
```sh
python -m startup.start_all_services
```

#### 1. Setup a listener and RabbitMQ object

In [1]:
from communication import protocol
from communication.rabbitmq import Rabbitmq
from threading import Thread
import numpy as np

def receiver() -> None:
    def on_error_message_received(channel, method, properties, body) -> None:
        print("\nERROR MESSAGE RECEIVED:")
        print(body)

    rmq = Rabbitmq(
        ip="localhost",
        port=5672,
        username="ur3e",
        password="ur3e",
        vhost="/",
        exchange="UR3E_AMQP",
        type="topic",
    )
    rmq.connect_to_server()
    rmq.subscribe(protocol.ROUTING_KEY_SIMPLE_ERROR_SERVICE, on_error_message_received)
    rmq.start_consuming()


Thread(target=receiver).start()

In [2]:
rmq = Rabbitmq(
    ip="localhost",
    port=5672,
    username="ur3e",
    password="ur3e",
    vhost="/",
    exchange="UR3E_AMQP",
    type="topic",
)
rmq.connect_to_server()

#### 2. Send a control message (without fault injection)

In [3]:
# unstuck previously stuck joints
rmq.send_message(
    routing_key=protocol.ROUTING_KEY_CTRL,
    message={
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.UNSTUCK_JOINT,
        protocol.CtrlMsgKeys.JOINTS: [0, 1, 2, 3, 4, 5],
    },
)

In [6]:
rmq.send_message(
    routing_key=protocol.ROUTING_KEY_CTRL,
    message={
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.LOAD_PROGRAM,
        protocol.CtrlMsgKeys.JOINT_POSITIONS: [
            [0.0, -np.pi / 2, np.pi / 2, -np.pi / 2, -np.pi / 2, 0.0]
            #[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        ],
        protocol.CtrlMsgKeys.MAX_VELOCITY: 60,
        protocol.CtrlMsgKeys.ACCELERATION: 80,
    },
)
rmq.send_message(
    routing_key=protocol.ROUTING_KEY_CTRL,
    message={
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.PLAY,
    },
)


ERROR MESSAGE RECEIVED:
{'status': False, 'actual_position': [-0.2985441212252103, -0.13105052433604733, 0.30329880366868806], 'simulated_position': [-0.29855000000000004, -0.13104999999999997, 0.30329999999999996], 'position_difference': [5.8787747897470766e-06, 5.243360473583536e-07, 1.1963313119034424e-06]}


#### 3. Send a control message with fault injection

In [7]:
rmq.send_message(
    routing_key=protocol.ROUTING_KEY_CTRL,
    message={
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.INJECT_FAULT,
        protocol.CtrlMsgKeys.FAULT_TYPE: protocol.FaultTypes.STUCK_JOINT,
        protocol.CtrlMsgKeys.JOINTS: [3, ],
        protocol.CtrlMsgKeys.DURATION: 60,
    },
)

rmq.send_message(
    routing_key=protocol.ROUTING_KEY_CTRL,
    message={
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.LOAD_PROGRAM,
        protocol.CtrlMsgKeys.JOINT_POSITIONS: [
            [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
        ],
        protocol.CtrlMsgKeys.MAX_VELOCITY: 60,
        protocol.CtrlMsgKeys.ACCELERATION: 80,
    },
)

rmq.send_message(
    routing_key=protocol.ROUTING_KEY_CTRL,
    message={
        protocol.CtrlMsgKeys.TYPE: protocol.CtrlMsgFields.PLAY,
    },
)


ERROR MESSAGE RECEIVED:
{'status': True, 'actual_position': [-0.5420911355088218, -0.2231390888070692, 0.15184323442208195], 'simulated_position': [-0.45675, -0.22315000000000002, 0.06650000000000002], 'position_difference': [0.08534113550882183, 1.0911192930812552e-05, 0.08534323442208193]}
